In [14]:
import sys
sys.path.append(r"C:\nirfasterFF")
import nirfasterff as ff # ff is short for fast and furious
import numpy as np
import matplotlib.pyplot as plt

In [15]:
fpert = 1.01

mua = 0.01
mua_pert = fpert*mua
musp = 0.33
musp_pert = fpert*musp
ri = 1.4

vol = 2 * np.ones((60,60,60))
vol = vol.astype(np.uint8)

meshing_params = ff.utils.MeshingParams(
    xPixelSpacing=0.5,  # Finer voxel resolution
    yPixelSpacing=0.5,
    SliceThickness=0.5,
    general_cell_size=1.0,  # Smaller elements
    cell_radius_edge=2.0,  # Better element quality
    facet_size=1.5,  # More refined surface
    facet_distance=1.0
)


# call the mesher
ele, nodes = ff.meshing.RunCGALMeshGenerator(vol, opt = meshing_params)

In [16]:
mesh = ff.base.stndmesh()
mesh.from_solid(ele, nodes)

# for each row, [region, mua(mm-1), musp(mm-1), ri]
prop = np.array([[1, 0.01, 0.330033, 1.33]])
mesh.set_prop(prop)
# set link (source-detector pairs)
mesh.link = np.array([[1,1,1]],dtype=np.int32)
# optodes
mesh.source = ff.base.optode(np.array([5,30,60]))
mesh.meas = ff.base.optode(np.array([55,30,60]))
# move them (for non-fixed) and calculate the integration function
mesh.touch_optodes()

touching sources
touching detectors


In [17]:
xgrid = np.arange(0., 60, 0.5)
ygrid = np.arange(0., 60, 0.5)
zgrid = np.arange(0., 60, 0.5)

mesh.gen_intmat(xgrid, ygrid, zgrid)

In [18]:
# TEST 1: comparing [perturbation model-doD vs mua-jacobian-doD] for normal mesh type, without US


J_mua = mesh.jacobian()[0] 

# creating duplicate mesh for measuring perturbation-model doD
mesh1 = ff.base.stndmesh()
mesh1.from_copy(mesh)


# change the optical properties of duplicated mesh1
mesh1.change_prop(-1, [1.01*mesh.mua[0], mesh.mus[0], mesh.ri[0]]) # 1% change in mua, as we have to compare with mua-Jacobian

# calculate the forward data and then the doD
data_mesh = mesh.femdata(0)[0]
data_mesh1 = mesh1.femdata(0)[0]
doD_pert_1 = np.log(data_mesh1.amplitude) - np.log(data_mesh.amplitude)

# Now calculate doD using the Jacobian method
dmua = 0.01*mesh.mua[0]*np.ones(xgrid.size * ygrid.size * zgrid.size)
doD_jacobian_1 = J_mua @ dmua

#Compare
print('\n')
print('doD from mua perturbation:',doD_pert_1)
print('doD from mua-Jacobian:',doD_jacobian_1)
big = max(abs(doD_pert_1),abs(doD_jacobian_1))
small = min(abs(doD_pert_1),abs(doD_jacobian_1))
print('percentage error:' ,((big-small)/big)*100)



Calculating direct field...
Calculating adjoint field...
Integrating...


doD from mua perturbation: [-0.02044762]
doD from mua-Jacobian: [-0.01967415]
percentage error: [3.78265081]


In [19]:
# TEST 2: comparing [perturbation model-doD vs mus-jacobian-doD] for normal mesh type, without US

J = mesh.jacobian(mus=True)[0]
# let's look only at the mus part
J_mus = J[:,:xgrid.size * ygrid.size * zgrid.size]

# creating duplicate mesh for measuring perturbation-model doD
mesh2 = ff.base.stndmesh()
mesh2.from_copy(mesh)


mesh2.change_prop(-1, [mesh.mua[0], 1.01*mesh.mus[0], mesh.ri[0]]) # 1% change in mus, as we have to compare with mus-Jacobian

# calculate the forward data and then the doD
data_mesh = mesh.femdata(0)[0]
data_mesh2 = mesh1.femdata(0)[0]
doD_pert_2 = np.log(data_mesh2.amplitude) - np.log(data_mesh.amplitude)

# Now calculate doD using the Jacobian method
dmus = 0.01*mesh.mus[0]*np.ones(xgrid.size * ygrid.size * zgrid.size)
doD_jacobian_2 = J_mus @ dmus

#Compare
print('\n')
print('doD from mus perturbation:',doD_pert_2)
print('doD from mus-Jacobian:',doD_jacobian_2)
big = max(abs(doD_pert_2),abs(doD_jacobian_2))
small = min(abs(doD_pert_2),abs(doD_jacobian_2))
print('percentage error:' ,((big-small)/big)*100)

Calculating direct field...
Calculating adjoint field...
Integrating...


doD from mus perturbation: [-0.02044762]
doD from mus-Jacobian: [-0.02510441]
percentage error: [18.54971398]


In [20]:
# Define US parameters to calulate Jacobian at tau = T/2

wv = 785 #nm
k0 = 2*np.pi / (wv/1e6)

#Define US parameters
fa = 5e6  # frequency of US (Hz)
va = 1480*1e3   # acoustic wave velocity
ka = 2*np.pi / (va/(fa))  # va=1480m/s
wa = 2*np.pi*fa   # 2*pi*fa
ltr = 1/mesh.mus
rho = 1000*1e-9   # for water
eta = 0.32       # for water
Sa = 1.
phi = 0.
P0 = 1e5  # US pressue(pascal)
tau = 0.5*(1/fa)  #T/2

# Define UltraSound position
x_cen,y_cen,z_cen = [30,30,30] #centre of the US , z-pos is measured from bottom surface(starting from 0) to top surface here in the mesh space
                               #(although in general terminology, its expressed as depth from top surface)

# cylinder
# dist = np.sqrt(np.square((mesh.nodes[:,0]-x_cen)) + np.square((mesh.nodes[:,1]-y_cen)) + np.square((mesh.nodes[:,2]-z_cen)))
# height = 7  #height of cylinder
# dia = 5     #diameter of cylinder
# ind= np.zeros(len(mesh.nodes))

# for i in range(len(mesh.nodes)):     # selecting nodes for Cylindrical US: Z-axis aligned   
#     if mesh.nodes[i,2]>(z_cen-height/2) and mesh.nodes[i,2]<(z_cen+height/2) and np.sqrt((mesh.nodes[i,0]-x_cen)**2+(mesh.nodes[i,1]-y_cen)**2) < dia/2 :
#         ind[i] = dist[i] 

# sphere
dia = 8
dist = np.sqrt(np.square((mesh.nodes[:,0]-x_cen)) + np.square((mesh.nodes[:,1]-y_cen)) + np.square((mesh.nodes[:,2]-z_cen)))
ind= np.zeros(len(mesh.nodes)) 
ind = dist < dia/2


selection = np.argwhere(ind != 0)
p0 = np.zeros(len(mesh.mua))
p0[selection] = P0/1e3           #assigning a pressure value to the selected nodes within the cylindrical vol(also converting into N/mm2)

ht = (np.square((p0*k0*mesh.ri)/(ka*rho*va*va))) * (1-np.cos(wa*tau)) * ( (np.square(eta))*(ka*ltr)*(np.arctan(ka*ltr)) + Sa*Sa/3 - 2*eta*Sa/np.cos(phi))


# mua_US = mua+musp*ht[0]
# mua_US_pertmua = mua_pert+musp*ht[0]
# mua_US_pertmus = mua+musp_pert*ht[0]

#print(ht[ht!=0])

In [21]:
# TEST 3: comparing [US-perturbed-perturbation model-doD  vs  US-perturbed-mua-jacobian-doD] for US-perturbed mesh 


# making a duplicate of main mesh for creating the 'US-perturbed-mesh'
mesh3 = ff.base.stndmesh()
mesh3.from_copy(mesh)


mesh3.mua = mesh.mua + mesh.mus*ht    # mesh3 is now the US-perturbed mesh, 
                                      # mesh3 will be used in all subsequent cases below which involves US.
mesh3.kappa = 1.0 / (3.0*(mesh3.mua + mesh3.mus))

J_mua_us = mesh3.jacobian()[0]    # US-mua-Jacobian

# creating duplicate of mesh3 for measuring US-perturbed-perturbation-model doD
mesh4 = ff.base.stndmesh()
mesh4.from_copy(mesh3)




# change the optical properties of duplicated mesh2
mesh4.change_prop(-1, [1.01*mesh.mua[0], mesh3.mus[0], mesh3.ri[0]]) # 1% change in mua, as we have to compare with mua-Jacobian

# calculate the forward data and then the doD
data_mesh3 = mesh3.femdata(0)[0]

mesh4.mua[selection] = mesh4.mua[0]+mesh.mus[0]*ht[0]
data_mesh4 = mesh4.femdata(0)[0]
doD_pert_3 = np.log(data_mesh4.amplitude) - np.log(data_mesh3.amplitude)





# Now calculate doD using the Jacobian method
dmua = 0.01*mesh3.mua[0]*np.ones(xgrid.size * ygrid.size * zgrid.size)
doD_jacobian_3 = J_mua_us @ dmua





#Compare
print('\n')
print('doD from US-perturbed-mua-perturbation model:',doD_pert_3)
print('doD from US-perturbed-mua-Jacobian:',doD_jacobian_3)
big = max(abs(doD_pert_3),abs(doD_jacobian_3))
small = min(abs(doD_pert_3),abs(doD_jacobian_3))
print('percentage error:' ,((big-small)/big)*100)

Calculating direct field...
Calculating adjoint field...
Integrating...


doD from US-perturbed-mua-perturbation model: [-0.01973156]
doD from US-perturbed-mua-Jacobian: [-0.01966679]
percentage error: [0.32825001]


In [22]:
# TEST 4: comparing [US-perturbed-perturbation model-doD  vs  US-perturbed-mus-jacobian-doD] for US-perturbed mesh

J1 = mesh3.jacobian(mus=True)[0]
# let's look only at the mus part
J_mus_us = J1[:,:xgrid.size * ygrid.size * zgrid.size]

# creating duplicate of mesh3 for measuring US-perturbed-perturbation-model doD
mesh5 = ff.base.stndmesh()
mesh5.from_copy(mesh3)   

# change the optical properties of duplicated mesh5
mesh5.change_prop(-1, [mesh.mua[0], 1.01*mesh.mus[0], mesh.ri[0]]) # 1% change in mus, as we have to compare with mus-Jacobian
             

# calculate the forward data and then the doD
data_mesh3 = mesh3.femdata(0)[0]
mesh5.mua[selection] = mesh.mua[0] + mesh5.mus[0]*ht[0]       # only for nodes inside US-cylinder, since 'mua = mua + mus*ht' for these nodes
                                                                  # this updation of mua is a consequence of the change in mus, 
                                                                  # since change in mus will reflect in mua for nodes inside us-cylinder
data_mesh5 = mesh5.femdata(0)[0]
doD_pert_4 = np.log(data_mesh5.amplitude) - np.log(data_mesh3.amplitude)




# Now calculate doD using the Jacobian method
dmus = 0.01*mesh3.mus[0]*np.ones(xgrid.size * ygrid.size * zgrid.size)
doD_jacobian_4 = J_mus @ dmus




#Compare
print('\n')
print('doD from mus perturbation:',doD_pert_4)
print('doD from mus-Jacobian:',doD_jacobian_4)
big = max(abs(doD_pert_4),abs(doD_jacobian_4))
small = min(abs(doD_pert_4),abs(doD_jacobian_4))
print('percentage error:' ,((big-small)/big)*100)

Calculating direct field...
Calculating adjoint field...
Integrating...


doD from mus perturbation: [-0.02457201]
doD from mus-Jacobian: [-0.02510441]
percentage error: [2.12076433]


In [23]:
# TEST 5: comparing [US-perturbed-perturbation model-doD  vs  US-perturbed-mua-jacobian-doD] for US-perturbed mesh
# Here, we add an absorption inclusion(a blob) at some point inside the block, instead of varying the entire absorption map by 1%

J_mua_us = mesh3.jacobian()[0]    # US-mua-Jacobian  # run the code block where mesh3 is initialized - (1, 1728000)



# creating duplicate of mesh3 for measuring US-perturbed-perturbation-model doD
mesh6 = ff.base.stndmesh()
mesh6.from_copy(mesh3)       # mua of mesh6 is 'mesh3.mua = mesh.mua + mesh.mus*ht' since it is derived from mesh3



# add a spherical absorption inclusion in mesh6
dia_ab = 8    #diameter of spherical absorber
x_cen_ab,y_cen_ab,z_cen_ab = [30,30,30]      #centre of the absorber
dist_ab = np.sqrt(np.square((mesh.nodes[:,0]-x_cen_ab)) + np.square((mesh.nodes[:,1]-y_cen_ab)) + np.square((mesh.nodes[:,2]-z_cen_ab)))
                                           #all meshes are derived from the original main 'mesh', hence its alright to use 'mesh.nodes' 
ind_ab = dist_ab < dia_ab/2
mua_selection = np.where(ind_ab)           #of all node points, this is the selection of nodes where the absorber inclusion exists





# Modify the absorption-map of mesh6(in mesh-space) by 1%, and calculate the forward data and sebsequently the perturbation doD
mesh6.mua[mua_selection] = 1.01*mesh6.mua[mua_selection] 

#!!!!! the below line should be turned ON only if the inclusion coincides with the US-position, else comment it out!!!!
mesh6.mua[mua_selection] = mesh6.mua[mua_selection] - 0.01*mesh6.mus[mua_selection]*ht[0]

mesh6.kappa[mua_selection] = 1.0 / (3.0*(mesh6.mua[mua_selection] + mesh6.mus[mua_selection]))

data_mesh3 = mesh3.femdata(0)[0]
data_mesh6 = mesh6.femdata(0)[0]
doD_pert_5 = np.log(data_mesh6.amplitude) - np.log(data_mesh3.amplitude)







# Now calculate doD using the Jacobian method
# creating the difference in absorption for selected nodes(to be multiplied with mua-Jacobian) in mesh-space and then interpolated to grid-space 
tmp1= np.zeros(len(mesh.nodes))    #all meshes are derived from the original main 'mesh', hence its alright to use 'mesh.nodes'
tmp1[mua_selection] = 0.01*mesh6.mua[mua_selection]  # this is the effective difference in abs (in mesh-space). a 1% change in 'just absorption' 
dmua = mesh.vol.mesh2grid@tmp1     # mesh to grid interpolation (difference in abs in grid-space) - (1728000,1)
doD_jacobian_5 = J_mua_us @ dmua





#Compare
print('\n')
print(f"doD from US-perturbed-absorption-blob-perturbation model at z = {60-z_cen_ab} mm:", doD_pert_5)
print(f"doD from US-perturbed-absorption-blob-Jacobian: model at z = {60-z_cen_ab} mm:", doD_jacobian_5)
big = max(abs(doD_pert_5),abs(doD_jacobian_5))
small = min(abs(doD_pert_5),abs(doD_jacobian_5))
print('percentage error:' ,((big-small)/big)*100)




Calculating direct field...
Calculating adjoint field...
Integrating...


doD from US-perturbed-absorption-blob-perturbation model at z = 30 mm: [-2.75532748e-05]
doD from US-perturbed-absorption-blob-Jacobian: model at z = 30 mm: [-2.7832513e-05]
percentage error: [1.00328051]


In [ ]:
# TEST 6: comparing [US-perturbed-perturbation model-doD  vs  US-perturbed-mus-jacobian-doD] for US-perturbed mesh
# Here, we add an scattering inclusion(a blob) at some point inside the block, instead of varying the entire mus-map by 1%

J2 = mesh3.jacobian(mus=True)[0]    
# let's look only at the mus part
J_mus_us_ = J2[:,:xgrid.size * ygrid.size * zgrid.size]    #(1, 1728000)

# creating duplicate of mesh3 for measuring US-perturbed-perturbation-model doD
mesh7 = ff.base.stndmesh()
mesh7.from_copy(mesh3)   


# add a spherical scatterer inclusion in mesh7
dia_ab = 8    #diameter of spherical scatterer
x_cen_ab,y_cen_ab,z_cen_ab = [30,30,30]      #centre of the scatterer
dist_ab = np.sqrt(np.square((mesh.nodes[:,0]-x_cen_ab)) + np.square((mesh.nodes[:,1]-y_cen_ab)) + np.square((mesh.nodes[:,2]-z_cen_ab)))
                                           #all meshes are derived from the original main 'mesh', hence its alright to use 'mesh.nodes' 
ind_ab = dist_ab < dia_ab/2
mus_selection = np.where(ind_ab)           #of all node points, this is the selection of nodes where the absorber inclusion exists




# Modify the scatterer-map of mesh7(in mesh-space) by 1%, and calculate the forward data and sebsequently the perturbation doD
mesh7.mus[mus_selection] = 1.01*mesh7.mus[mus_selection] 

#!!!!! the below line should be turned ON only if the inclusion coincides with the US-position, else comment it out!!!! 
mesh7.mua[mus_selection] = mesh7.mua[mus_selection] + mesh7.mus[mus_selection]*ht[0]  # since, mua inside the US-cylinder also depends on mus
                                                                                          
mesh7.kappa[mus_selection] = 1.0 / (3.0*(mesh7.mua[mus_selection] + mesh7.mus[mus_selection]))
data_mesh3 = mesh3.femdata(0)[0]
data_mesh7 = mesh7.femdata(0)[0]
doD_pert_6 = np.log(data_mesh7.amplitude) - np.log(data_mesh3.amplitude)




# Now calculate doD using the Jacobian method
# creating the difference in scattering for selected nodes(to be multiplied with mus-Jacobian) in mesh-space and then mapping it to grid-space 
tmp1= np.zeros(len(mesh.nodes))    #all meshes are derived from the original main 'mesh', hence its alright to use 'mesh.nodes'
tmp1[mus_selection] = 0.01*mesh3.mus[mus_selection]    # this is the effective difference in mus (in mesh-space). a 1% change in 'just scattering' 
dmus = mesh.vol.mesh2grid@tmp1     # mesh to grid interpolation (difference in scattering in grid-space) - (1728000,1)
doD_jacobian_6 = J_mus_us_ @ dmus




#Compare
print('\n')
print(f"doD from US-perturbed-scatterer-blob-perturbation model at z = {60-z_cen_ab} mm:", doD_pert_6)
print(f"doD from US-perturbed-scatterer-blob-Jacobian: model at z = {60-z_cen_ab} mm:", doD_jacobian_6)
big = max(abs(doD_pert_6),abs(doD_jacobian_6))
small = min(abs(doD_pert_6),abs(doD_jacobian_6))
print('percentage error:' ,((big-small)/big)*100)

Calculating direct field...
Calculating adjoint field...
Integrating...


doD from US-perturbed-scatterer-blob-perturbation model at z = 30 mm: [2.14246757e-07]
doD from US-perturbed-scatterer-blob-Jacobian: model at z = 30 mm: [2.47931678e-07]
percentage error: [13.58637236]


In [12]:
# TEST 7: comparing [US-perturbed-perturbation model-doD  vs  US-perturbed-mua-jacobian-doD] for US-perturbed mesh
# Here, we perform an absorption inclusion(a blob) z-scan inside the block

J_mua_us = mesh3.jacobian()[0]    # US-mua-Jacobian  # run the code block where mesh3 is initialized - (1, 1728000)

# creating duplicate of mesh3 for measuring US-perturbed-perturbation-model doD
mesh8 = ff.base.stndmesh()

for i in range(10,51,10):
   mesh8.from_copy(mesh3)       # mua of mesh8 is 'mesh3.mua = mesh.mua + mesh.mus*ht' since it is derived from mesh3

   # add a spherical absorption inclusion in mesh8
   dia_ab =  6   #diameter of spherical absorber
   x_cen_ab,y_cen_ab,z_cen_ab = [30,30,i]      #centre of the absorber
   dist_ab = np.sqrt(np.square((mesh.nodes[:,0]-x_cen_ab)) + np.square((mesh.nodes[:,1]-y_cen_ab)) + np.square((mesh.nodes[:,2]-z_cen_ab)))
                                             #all meshes are derived from the original main 'mesh', hence its alright to use 'mesh.nodes' 
   ind_ab = dist_ab < dia_ab/2
   mua_selection = np.where(ind_ab)           #of all node points, this is the selection of nodes where the absorber inclusion exists

   



   # Modify the absorption-map of mesh8(in mesh-space) by 1%, and calculate the forward data and sebsequently the perturbation doD
   mesh8.mua[mua_selection] = 1.01*mesh8.mua[mua_selection] 

   if i==30:                                 #only if the inclusion coincides with the US-position!
      mesh8.mua[mua_selection] = mesh8.mua[mua_selection] - 0.01*mesh8.mus[mua_selection]*ht[0]

   mesh8.kappa[mua_selection] = 1.0 / (3.0*(mesh8.mua[mua_selection] + mesh8.mus[mua_selection]))
   data_mesh3 = mesh3.femdata(0)[0]
   data_mesh8 = mesh8.femdata(0)[0]
   doD_pert_7 = np.log(data_mesh8.amplitude) - np.log(data_mesh3.amplitude)



   # Now calculate doD using the Jacobian method
   # creating the difference in absorption for selected nodes(to be multiplied with mua-Jacobian) in mesh-space and then interpolated to grid-space 
   tmp1= np.zeros(len(mesh.nodes))    #all meshes are derived from the original main 'mesh', hence its alright to use 'mesh.nodes'
   tmp1[mua_selection] = 0.01*mesh8.mua[mua_selection]         # this is the effective difference in abs (in mesh-space). a 1% change in 'just absorption' 
   dmua = mesh.vol.mesh2grid@tmp1     # mesh to grid interpolation (difference in abs in grid-space) - (1728000,1)
   doD_jacobian_7 = J_mua_us @ dmua


   #Compare
   print('\n')
   print(f"doD from US-perturbed-absorption-blob-perturbation model at z = {60-i} mm:", doD_pert_7)
   print(f"doD from US-perturbed-absorption-blob-Jacobian: model at z = {60-i} mm:", doD_jacobian_7)
   #big = max(abs(doD_pert_7),abs(doD_jacobian_7))
   #small = min(abs(doD_pert_7),abs(doD_jacobian_7))
   print('percentage error:' ,abs((doD_jacobian_7-doD_pert_7)/doD_pert_7)*100)

Calculating direct field...
Calculating adjoint field...
Integrating...


doD from US-perturbed-absorption-blob-perturbation model at z = 50 mm: [-1.04326091e-07]
doD from US-perturbed-absorption-blob-Jacobian: model at z = 50 mm: [-1.09162767e-07]
percentage error: [4.63611384]


doD from US-perturbed-absorption-blob-perturbation model at z = 40 mm: [-1.07433421e-06]
doD from US-perturbed-absorption-blob-Jacobian: model at z = 40 mm: [-1.10683042e-06]
percentage error: [3.02477656]


doD from US-perturbed-absorption-blob-perturbation model at z = 30 mm: [-1.0629439e-05]
doD from US-perturbed-absorption-blob-Jacobian: model at z = 30 mm: [-1.07387553e-05]
percentage error: [1.02842958]


doD from US-perturbed-absorption-blob-perturbation model at z = 20 mm: [-4.8503479e-05]
doD from US-perturbed-absorption-blob-Jacobian: model at z = 20 mm: [-4.74379611e-05]
percentage error: [2.19678661]


doD from US-perturbed-absorption-blob-perturbation model at z = 10 mm: [-0.00010757]
doD from US

In [13]:
# TEST 8: comparing [US-perturbed-perturbation model-doD  vs  US-perturbed-mus-jacobian-doD] for US-perturbed mesh
# Here, we add an scattering inclusion(a blob) z-scan inside the block

J3 = mesh3.jacobian(mus=True)[0]    
# let's look only at the mus part
J_mus_us_ = J3[:,:xgrid.size * ygrid.size * zgrid.size]    #(1, 1728000)


# creating duplicate of mesh3 for measuring US-perturbed-perturbation-model doD
mesh9 = ff.base.stndmesh()

for i in range(10,51,10):
   mesh9.from_copy(mesh3)       # mua of mesh9 is 'mesh3.mua = mesh.mua + mesh.mus*ht' since it is derived from mesh3

   # add a spherical absorption inclusion in mesh9
   dia_ab = 8    #diameter of spherical scatterer
   x_cen_ab,y_cen_ab,z_cen_ab = [30,30,i]      #centre of the absorber
   dist_ab = np.sqrt(np.square((mesh.nodes[:,0]-x_cen_ab)) + np.square((mesh.nodes[:,1]-y_cen_ab)) + np.square((mesh.nodes[:,2]-z_cen_ab)))
                                             #all meshes are derived from the original main 'mesh', hence its alright to use 'mesh.nodes' 
   ind_ab = dist_ab < dia_ab/2
   mus_selection = np.where(ind_ab)           #of all node points, this is the selection of nodes where the absorber inclusion exists




   # Modify the scatterer-map of mesh9(in mesh-space) by 1%, and calculate the forward data and sebsequently the perturbation doD
   mesh9.mus[mus_selection] = 1.01*mesh9.mus[mus_selection] 

   if i==30:                                  #only if the inclusion coincides with the US-position!
      mesh9.mua[mus_selection] = mesh9.mua[mus_selection] + mesh9.mus[mus_selection] *ht[0]  # since, mua inside the US-cylinder also depends on mus
                                                                                             

   
   mesh9.kappa[mus_selection] = 1.0 / (3.0*(mesh9.mua[mus_selection] + mesh9.mus[mus_selection]))
   data_mesh3 = mesh3.femdata(0)[0]
   data_mesh9 = mesh9.femdata(0)[0]
   doD_pert_8 = np.log(data_mesh9.amplitude) - np.log(data_mesh3.amplitude)


   # Now calculate doD using the Jacobian method
   # creating the difference in scattering for selected nodes(to be multiplied with mus-Jacobian) in mesh-space and then mapping it to grid-space 
   tmp1= np.zeros(len(mesh.nodes))    #all meshes are derived from the original main 'mesh', hence its alright to use 'mesh.nodes'
   tmp1[mus_selection] = 0.01*mesh9.mus[mus_selection]    # this is the effective difference in mus (in mesh-space). a 1% change in 'just scattering' 
   dmus = mesh.vol.mesh2grid@tmp1     # mesh to grid interpolation (difference in scattering in grid-space) - (1728000,1)
   doD_jacobian_8 = J_mus_us_ @ dmus



   #Compare
   print('\n')
   print(f"doD from US-perturbed-scatterer-blob-perturbation model at z = {60-i} mm:", doD_pert_8)
   print(f"doD from US-perturbed-scatterer-blob-Jacobian model at z = {60-i} mm:", doD_jacobian_8)
   #big = max(abs(doD_pert_8),abs(doD_jacobian_8))
   #small = min(abs(doD_pert_8),abs(doD_jacobian_8))
   print('percentage error:' ,abs((doD_jacobian_8-doD_pert_8)/doD_pert_8)*100)

Calculating direct field...
Calculating adjoint field...
Integrating...


doD from US-perturbed-scatterer-blob-perturbation model at z = 50 mm: [2.99229885e-07]
doD from US-perturbed-scatterer-blob-Jacobian model at z = 50 mm: [3.0423833e-07]
percentage error: [1.67377835]


doD from US-perturbed-scatterer-blob-perturbation model at z = 40 mm: [1.62685487e-06]
doD from US-perturbed-scatterer-blob-Jacobian model at z = 40 mm: [1.65510562e-06]
percentage error: [1.73652577]


doD from US-perturbed-scatterer-blob-perturbation model at z = 30 mm: [2.14247315e-07]
doD from US-perturbed-scatterer-blob-Jacobian model at z = 30 mm: [2.50410995e-07]
percentage error: [16.87940874]


doD from US-perturbed-scatterer-blob-perturbation model at z = 20 mm: [-0.00012502]
doD from US-perturbed-scatterer-blob-Jacobian model at z = 20 mm: [-0.00012729]
percentage error: [1.81786245]


doD from US-perturbed-scatterer-blob-perturbation model at z = 10 mm: [-0.00056444]
doD from US-perturbed-scatterer-blob